In [3]:


# ── stdlib ────────────────────────────────────────────────────────────────────
import os, glob, warnings, logging
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s │ %(levelname)s │ %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger("MGT")

# ── numerical / ML ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold


In [4]:
from econml.dml import LinearDML

ImportError: Numba needs NumPy 2.3 or less. Got NumPy 2.4.

In [19]:
# ── causal inference ──────────────────────────────────────────────────────────
try:
    import dowhy
    from dowhy import CausalModel
    HAS_DOWHY = True
except ImportError:
    HAS_DOWHY = False
    log.warning("DoWhy not installed – causal estimation stage will be skipped. "
                "pip install dowhy econml")

try:
    from econml.dml import LinearDML
    HAS_ECONML = True
except ImportError:
    HAS_ECONML = False

# ── deep learning ─────────────────────────────────────────────────────────────
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader as TorchLoader
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    HAS_TORCH = True
    log.info(f"PyTorch device: {DEVICE}")
except ImportError:
    HAS_TORCH = False
    log.warning("PyTorch not installed – LSTM stage will be skipped. pip install torch")


# ══════════════════════════════════════════════════════════════════════════════
#  0.  CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

from dataclasses import dataclass
# Update: set data_dir for running from notebooks folder
@dataclass
class Config:
    # ── paths ─────────────────────────────────────────────────────────────────
    data_dir:        str   = "../data"          # root; expects train/ and test/ sub-dirs
    output_dir:      str   = "../outputs"
    model_dir:       str   = "../models"

    # ── feature engineering ───────────────────────────────────────────────────
    power_lags:      int   = 8               # how many el_power lags to include
    voltage_lags:    int   = 4               # how many input_voltage lags to include
    seq_len:         int   = 32              # LSTM look-back window (timesteps)

    # ── DoWhy / causal ────────────────────────────────────────────────────────
    treatment_col:   str   = "input_voltage"
    outcome_col:     str   = "el_power"
    n_refutations:   int   = 3               # placebo trials

    # ── synthetic interventions ───────────────────────────────────────────────
    n_synthetic_runs:int   = 4               # extra simulated experiments
    voltage_range:   tuple = (8.0, 14.0)    # volts – range for counterfactual V

    # ── LSTM training ─────────────────────────────────────────────────────────
    hidden_size:     int   = 64
    num_layers:      int   = 2
    dropout:         float = 0.25
    epochs:          int   = 60
    batch_size:      int   = 128
    lr:              float = 1e-3
    patience:        int   = 10              # early-stop patience

    # ── misc ──────────────────────────────────────────────────────────────────
    seed:            int   = 42
    verbose:         bool  = True

CFG = Config()

import numpy as np
np.random.seed(CFG.seed)
if HAS_TORCH:
    torch.manual_seed(CFG.seed)

for d in [CFG.output_dir, CFG.model_dir]:
    Path(d).mkdir(parents=True, exist_ok=True)

18:46:47 │ INFO │ PyTorch device: cpu


In [20]:

# ══════════════════════════════════════════════════════════════════════════════
#  1.  DATA LOADING & FEATURE ENGINEERING
# ══════════════════════════════════════════════════════════════════════════════

def load_experiment_csv(path: str) -> pd.DataFrame:
    """
    Load a single experiment CSV.

    Handles two common layouts:
      A) comma-separated with a named timestamp index column
      B) tab-separated / unnamed first column (raw export)
    """

    

    df = pd.read_csv(path, index_col=0, parse_dates=True)
    df.index.name = "timestamp"
    # keep only the two model columns
    df = df[["input_voltage", "el_power"]].copy()
    df = df.astype(float)
    df = df.dropna()
    return df


def load_all_experiments(split: str = "train") -> List[pd.DataFrame]:
    """
    Load all CSVs from data/<split>/ and return a list of DataFrames.
    Each DataFrame carries a .name attribute with the experiment id.
    """
    pattern = os.path.join(CFG.data_dir, split, "*.csv")
    paths = sorted(glob.glob(pattern))
    if not paths:
        log.warning(f"No CSV files found at {pattern}. "
                    "Using synthetic demo data.")
        return _generate_demo_experiments(n=6 if split == "train" else 2)

    experiments = []
    for p in paths:
        df = load_experiment_csv(p)
        df.attrs["name"] = Path(p).stem
        experiments.append(df)
        log.info(f"  Loaded {Path(p).name:30s} → {len(df):,} rows  "
                 f"V={df.input_voltage.unique()}")
    return experiments


def _generate_demo_experiments(n: int) -> List[pd.DataFrame]:
    """
    Synthetic stand-in when real data are absent.
    el_power ~ AR(2) + β·voltage + noise
    """
    dfs = []
    voltage_levels = np.linspace(9.0, 13.0, n)
    for i, v in enumerate(voltage_levels):
        T = 9_000
        power = np.zeros(T)
        power[0] = 1200 + 60 * v
        power[1] = 1200 + 60 * v
        for t in range(2, T):
            power[t] = (0.55 * power[t-1] + 0.25 * power[t-2]
                        + 80 * v + np.random.normal(0, 12))
        df = pd.DataFrame({
            "input_voltage": v,
            "el_power":      power,
        })
        df.attrs["name"] = f"experiment_{i+1}"
        dfs.append(df)
    return dfs


def make_lag_features(df: pd.DataFrame,
                      power_lags: int = CFG.power_lags,
                      voltage_lags: int = CFG.voltage_lags) -> pd.DataFrame:
    """
    Expand a 2-column df into a supervised dataset with temporal lag features.

    Causal ordering enforced:
      • el_power(t)      = outcome
      • input_voltage(t-1..L_v) = treatment lags  (no current voltage → no leakage)
      • el_power(t-1..L_p)     = autoregressive confounders
    """
    out = df.copy()

    # autoregressive power lags
    for k in range(1, power_lags + 1):
        out[f"power_lag{k}"] = df["el_power"].shift(k)

    # voltage lags (strictly causal: t-1 is earliest treatment effect visible at t)
    for k in range(1, voltage_lags + 1):
        out[f"voltage_lag{k}"] = df["input_voltage"].shift(k)

    out = out.dropna().reset_index(drop=True)
    return out


def build_regression_frame(experiments: List[pd.DataFrame]) -> pd.DataFrame:
    """Stack all experiments into one regression-ready DataFrame."""
    frames = []
    for df in experiments:
        feat = make_lag_features(df)
        feat["experiment"] = df.attrs.get("name", "unknown")
        frames.append(feat)
    combined = pd.concat(frames, ignore_index=True)
    log.info(f"Regression frame: {combined.shape[0]:,} rows × {combined.shape[1]} cols")
    return combined


# ══════════════════════════════════════════════════════════════════════════════
#  2.  CAUSAL DAG – VISUALISATION & DEFINITION
# ══════════════════════════════════════════════════════════════════════════════

def build_causal_dag() -> Tuple[nx.DiGraph, str]:
    """
    Return the NetworkX DAG and its GML string for DoWhy.

    DAG structure
    ─────────────
       input_voltage(t-1) ──────────────────────────────────┐
       input_voltage(t-2) ──────────────────────────────────┤
       ...                                                   ▼
       el_power(t-1)  ──► el_power(t-2)  ──► …   ──►   el_power(t)
         │                                               ▲
         └───────────────────────────────────────────────┘
    """
    G = nx.DiGraph()

    # nodes
    G.add_node("el_power_t",  label="el_power(t)",   kind="outcome")
    for k in range(1, CFG.power_lags + 1):
        G.add_node(f"power_lag{k}",   label=f"el_power(t-{k})",   kind="confounder")
    for k in range(1, CFG.voltage_lags + 1):
        G.add_node(f"voltage_lag{k}", label=f"voltage(t-{k})", kind="treatment")

    # treatment → outcome
    for k in range(1, CFG.voltage_lags + 1):
        G.add_edge(f"voltage_lag{k}", "el_power_t")

    # AR chain (autoregressive path)
    for k in range(1, CFG.power_lags + 1):
        G.add_edge(f"power_lag{k}", "el_power_t")
    for k in range(1, CFG.power_lags):
        G.add_edge(f"power_lag{k+1}", f"power_lag{k}")  # ordering

    # GML for DoWhy
    gml_lines = ["graph [", "  directed 1"]
    for i, n in enumerate(G.nodes()):
        gml_lines.append(f'  node [ id {i} label "{n}" ]')
    node_idx = {n: i for i, n in enumerate(G.nodes())}
    for u, v in G.edges():
        gml_lines.append(f'  edge [ source {node_idx[u]} target {node_idx[v]} ]')
    gml_lines.append("]")
    gml_string = "\n".join(gml_lines)

    return G, gml_string


def plot_dag(G: nx.DiGraph, save_path: str) -> None:
    """Render and save the causal DAG."""
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.set_facecolor("#0f1117")
    fig.patch.set_facecolor("#0f1117")

    color_map = {"outcome": "#e63946", "treatment": "#457b9d", "confounder": "#2a9d8f"}
    node_colors = [color_map.get(G.nodes[n].get("kind", "confounder"), "#aaa")
                   for n in G.nodes()]

    pos = {}
    outcome_nodes  = [n for n in G.nodes() if G.nodes[n].get("kind") == "outcome"]
    treatment_nodes= [n for n in G.nodes() if G.nodes[n].get("kind") == "treatment"]
    confounder_nodes=[n for n in G.nodes() if G.nodes[n].get("kind") == "confounder"]

    for i, n in enumerate(treatment_nodes):
        pos[n] = (0, (i - len(treatment_nodes)/2) * 1.2)
    for i, n in enumerate(confounder_nodes):
        pos[n] = (2, (i - len(confounder_nodes)/2) * 1.2)
    for n in outcome_nodes:
        pos[n] = (4, 0)

    nx.draw_networkx(G, pos=pos, ax=ax,
                     node_color=node_colors,
                     node_size=1100,
                     font_size=7,
                     font_color="white",
                     edge_color="#888",
                     arrows=True,
                     arrowsize=15,
                     width=1.4)

    legend = [mpatches.Patch(color=v, label=k) for k, v in color_map.items()]
    ax.legend(handles=legend, loc="lower left",
              facecolor="#1a1a2e", edgecolor="#444", labelcolor="white", fontsize=9)
    ax.set_title("MGT Causal DAG  –  el_power(t) ← voltage history + power history",
                 color="white", fontsize=11, pad=12)
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()
    log.info(f"DAG saved → {save_path}")


# ══════════════════════════════════════════════════════════════════════════════
#  3.  DOWHY CAUSAL ESTIMATION  (G-Formula / ITE + Refutation)
# ══════════════════════════════════════════════════════════════════════════════

def run_dowhy_estimation(reg_frame: pd.DataFrame,
                         gml_string: str) -> Optional[Dict]:
    """
    Estimate the Average Treatment Effect (ATE) of input_voltage on el_power
    using DoWhy + EconML LinearDML as the statistical estimator.

    Returns a results dict including:
      • ate      – point estimate
      • refutation_results – list of placebo outcomes (should be ≈ 0)
    """
    if not (HAS_DOWHY and HAS_ECONML):
        log.warning("Skipping DoWhy – missing dowhy/econml packages.")
        return None

    log.info("─── DoWhy Causal Estimation ───────────────────────────────────")

    # DoWhy requires a "treatment" column (binary or continuous).
    # We use voltage_lag1 as the primary treatment proxy.
    df_causal = reg_frame.copy()
    treatment  = "voltage_lag1"
    outcome    = CFG.outcome_col
    confounders= [c for c in df_causal.columns
                  if c.startswith("power_lag") and c != outcome]

    # Sub-sample for speed (10 k rows max)
    if len(df_causal) > 10_000:
        df_causal = df_causal.sample(10_000, random_state=CFG.seed)

    model = CausalModel(
        data=df_causal,
        treatment=treatment,
        outcome=outcome,
        common_causes=confounders,
        graph=gml_string,
    )

    # Identify the causal effect
    identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
    log.info(f"  Estimand: {identified_estimand.estimands}")

    # ── Estimate with Linear DML (orthogonal / double-ML) ────────────────────
    estimate = model.estimate_effect(
        identified_estimand,
        method_name="backdoor.econml.dml.LinearDML",
        method_params={
            "init_params": {
                "model_y": GradientBoostingRegressor(n_estimators=80,
                                                     max_depth=4,
                                                     random_state=CFG.seed),
                "model_t": GradientBoostingRegressor(n_estimators=80,
                                                     max_depth=3,
                                                     random_state=CFG.seed),
                "featurizer": None,
                "linear_first_stages": False,
                "cv": 3,
                "random_state": CFG.seed,
            },
            "fit_params": {}
        }
    )

    ate = float(estimate.value)
    log.info(f"  ATE (voltage → power): {ate:+.4f} W / V")

    # ── Placebo Treatment Refutation ─────────────────────────────────────────
    log.info("  Running Placebo Treatment Refutation …")
    refutation_results = []
    for trial in range(CFG.n_refutations):
        ref = model.refute_estimate(
            identified_estimand,
            estimate,
            method_name="placebo_treatment_refuter",
            placebo_type="permute",
            num_simulations=5,
        )
        placebo_ate = float(ref.new_effect)
        passed = abs(placebo_ate) < abs(ate) * 0.15
        refutation_results.append({
            "trial":      trial + 1,
            "placebo_ate": placebo_ate,
            "original_ate": ate,
            "passed":     passed,
        })
        log.info(f"    Trial {trial+1}: placebo ATE={placebo_ate:+.4f}  "
                 f"{'✓ PASS' if passed else '✗ FAIL (possible overfit)'}")

    return {
        "ate":               ate,
        "estimand":          str(identified_estimand),
        "refutation_results": refutation_results,
        "model":             model,
        "estimate":          estimate,
    }


# ══════════════════════════════════════════════════════════════════════════════
#  4.  SYNTHETIC INTERVENTIONS  (counterfactual augmentation)
# ══════════════════════════════════════════════════════════════════════════════

class ResponseSurface:
    """
    A lightweight ridge-regression model that learns
        el_power(t) = f(voltage_lag1..L, power_lag1..L)
    and is used to *simulate* counterfactual runs at new voltage levels.
    """

    def __init__(self):
        self.scaler_X = StandardScaler()
        self.scaler_y = StandardScaler()
        self.model    = Ridge(alpha=1.0)
        self._feature_cols: List[str] = []

    def fit(self, reg_frame: pd.DataFrame) -> "ResponseSurface":
        self._feature_cols = [c for c in reg_frame.columns
                              if c.startswith(("power_lag", "voltage_lag"))]
        X = reg_frame[self._feature_cols].values
        y = reg_frame[CFG.outcome_col].values.reshape(-1, 1)
        self.model.fit(
            self.scaler_X.fit_transform(X),
            self.scaler_y.fit_transform(y).ravel()
        )
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.scaler_y.inverse_transform(
            self.model.predict(self.scaler_X.transform(X)).reshape(-1, 1)
        ).ravel()


def generate_synthetic_interventions(
        real_experiments: List[pd.DataFrame],
        response: ResponseSurface,
        n_runs: int = CFG.n_synthetic_runs,
) -> List[pd.DataFrame]:
    """
    For each synthetic run:
      1. Sample a random voltage pattern (could be ramp, step, sine, noise)
      2. Roll the response model forward autoregressively to get el_power
      3. Return a full DataFrame matching the real experiment format

    This prevents the model from overfitting to the limited set of observed
    voltage levels by injecting structural variation.
    """
    T = len(real_experiments[0])
    synthetic = []

    voltage_patterns = {
        "ramp":     lambda t: np.linspace(*CFG.voltage_range, T),
        "step":     lambda t: np.where(t < T//2,
                                       CFG.voltage_range[0],
                                       CFG.voltage_range[1]).astype(float),
        "sine":     lambda t: (np.mean(CFG.voltage_range)
                               + np.ptp(CFG.voltage_range)/4
                               * np.sin(2*np.pi*t/(T//4))),
        "noisy_flat": lambda t: (np.random.choice(np.linspace(*CFG.voltage_range, 8))
                                  + np.random.normal(0, 0.3, T)),
    }

    pattern_names = list(voltage_patterns.keys())

    for i in range(n_runs):
        pname  = pattern_names[i % len(pattern_names)]
        t_arr  = np.arange(T)
        voltage = voltage_patterns[pname](t_arr)
        voltage = np.clip(voltage, *CFG.voltage_range)

        # Seed power from the mean of real experiments
        mean_power_init = np.mean([e["el_power"].values[:CFG.power_lags + 1]
                                   for e in real_experiments], axis=0)
        power = list(mean_power_init) + [np.nan] * (T - len(mean_power_init))

        # Autoregressive rollout
        for t in range(CFG.power_lags, T):
            row_dict = {}
            for k in range(1, CFG.power_lags + 1):
                row_dict[f"power_lag{k}"] = power[t - k]
            for k in range(1, CFG.voltage_lags + 1):
                row_dict[f"voltage_lag{k}"] = voltage[max(0, t - k)]
            x = np.array([[row_dict[c] for c in response._feature_cols]])
            power[t] = float(response.predict(x)[0])
            # add small autoregressive noise to prevent degenerate trajectories
            power[t] += np.random.normal(0, 5)

        df_syn = pd.DataFrame({
            "input_voltage": voltage,
            "el_power":      power,
        }).dropna()
        df_syn.attrs["name"] = f"synthetic_{pname}_{i+1}"
        df_syn.attrs["is_synthetic"] = True
        synthetic.append(df_syn)
        log.info(f"  Synthetic run '{pname}': {len(df_syn):,} rows  "
                 f"V∈[{voltage.min():.1f}, {voltage.max():.1f}]  "
                 f"P̄={np.nanmean(power):.1f} W")

    return synthetic


# ══════════════════════════════════════════════════════════════════════════════
#  5.  CAUSAL LSTM  (Granger-causality constrained)
# ══════════════════════════════════════════════════════════════════════════════

if HAS_TORCH:

    class MGTSequenceDataset(Dataset):
        """
        Sliding-window dataset over a list of experiments.
        Each item: (X_seq, y_scalar)
          X_seq  shape [seq_len, 2]  – [input_voltage, el_power] at t-seq_len..t-1
          y      shape []             – el_power at t
        """
        def __init__(self, experiments: List[pd.DataFrame], seq_len: int = CFG.seq_len):
            self.seq_len = seq_len
            self.samples: List[Tuple[np.ndarray, float]] = []
            for df in experiments:
                vals = df[["input_voltage", "el_power"]].values.astype(np.float32)
                for i in range(seq_len, len(vals)):
                    x = vals[i - seq_len: i]          # shape [seq_len, 2]
                    y = vals[i, 1]                     # el_power at t
                    self.samples.append((x, y))

        def __len__(self):  return len(self.samples)
        def __getitem__(self, idx):
            x, y = self.samples[idx]
            return torch.tensor(x), torch.tensor(y)


    class CausalLSTM(nn.Module):
        """
        Neural Granger Causality LSTM.

        Causal constraint:
          • Both channels (voltage, power) feed into the shared LSTM – this lets
            the network model the full joint dynamics.
          • The output head is fitted only on the hidden states; no future voltage
            is ever exposed during prediction.
          • Interpretability: the `granger_mask` identifies which input channels
            contribute most (tracked during ablation analysis).

        Dropout regularisation + LayerNorm are critical given the small N_experiments.
        """

        def __init__(self,
                     input_size: int  = 2,        # [voltage, power]
                     hidden_size: int = CFG.hidden_size,
                     num_layers: int  = CFG.num_layers,
                     dropout: float   = CFG.dropout):
            super().__init__()

            self.hidden_size = hidden_size
            self.num_layers  = num_layers

            self.input_norm = nn.LayerNorm(input_size)

            self.lstm = nn.LSTM(
                input_size  = input_size,
                hidden_size = hidden_size,
                num_layers  = num_layers,
                batch_first = True,
                dropout     = dropout if num_layers > 1 else 0.0,
            )

            self.head = nn.Sequential(
                nn.LayerNorm(hidden_size),
                nn.Dropout(dropout),
                nn.Linear(hidden_size, hidden_size // 2),
                nn.GELU(),
                nn.Linear(hidden_size // 2, 1),
            )

        def forward(self, x: "torch.Tensor") -> "torch.Tensor":
            """
            x: [B, seq_len, 2]
            returns: [B]  (el_power predictions)
            """
            x = self.input_norm(x)
            out, _ = self.lstm(x)                    # [B, seq_len, hidden]
            last    = out[:, -1, :]                  # take final timestep
            pred    = self.head(last).squeeze(-1)    # [B]
            return pred

        def granger_ablation(self,
                             x: "torch.Tensor",
                             channel: int) -> "torch.Tensor":
            """
            Zero out one input channel and return predictions.
            If zeroing voltage channel causes a large drop in accuracy,
            voltage Granger-causes power in this model.
            """
            x_abl = x.clone()
            x_abl[:, :, channel] = 0.0
            return self.forward(x_abl)


    class EarlyStopping:
        def __init__(self, patience: int = CFG.patience, delta: float = 1e-4):
            self.patience = patience; self.delta = delta
            self.best = np.inf; self.counter = 0; self.stop = False

        def __call__(self, val_loss: float):
            if val_loss < self.best - self.delta:
                self.best = val_loss; self.counter = 0
            else:
                self.counter += 1
                if self.counter >= self.patience:
                    self.stop = True


    def train_causal_lstm(
            train_experiments: List[pd.DataFrame],
            val_fraction: float = 0.15,
    ) -> Tuple["CausalLSTM", StandardScaler, List[float], List[float]]:
        """
        Train the CausalLSTM with:
          • StandardScaler on both inputs (fit only on train)
          • 85/15 train/val split within the pooled dataset
          • AdamW + CosineAnnealingLR scheduler
          • Early stopping
        """
        log.info("─── Training Causal LSTM ───────────────────────────────────")

        # ── scale ─────────────────────────────────────────────────────────────
        all_vals = np.vstack([e[["input_voltage", "el_power"]].values
                              for e in train_experiments])
        scaler = StandardScaler()
        scaler.fit(all_vals)

        scaled_experiments = []
        for df in train_experiments:
            df_s = df.copy()
            df_s[["input_voltage", "el_power"]] = scaler.transform(
                df[["input_voltage", "el_power"]].values)
            df_s.attrs = df.attrs
            scaled_experiments.append(df_s)

        # ── dataset split ─────────────────────────────────────────────────────
        dataset = MGTSequenceDataset(scaled_experiments, CFG.seq_len)
        n_val   = max(1, int(len(dataset) * val_fraction))
        n_train = len(dataset) - n_val
        train_ds, val_ds = torch.utils.data.random_split(
            dataset, [n_train, n_val],
            generator=torch.Generator().manual_seed(CFG.seed)
        )

        train_dl = TorchLoader(train_ds, batch_size=CFG.batch_size,
                               shuffle=True,  drop_last=True)
        val_dl   = TorchLoader(val_ds,   batch_size=CFG.batch_size,
                               shuffle=False, drop_last=False)

        log.info(f"  Train samples: {n_train:,}  │  Val samples: {n_val:,}")

        # ── model / optimiser ─────────────────────────────────────────────────
        model     = CausalLSTM().to(DEVICE)
        optimiser = torch.optim.AdamW(model.parameters(),
                                      lr=CFG.lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimiser, T_max=CFG.epochs, eta_min=CFG.lr * 0.05)
        criterion = nn.HuberLoss(delta=1.0)
        stopper   = EarlyStopping(CFG.patience)

        train_losses, val_losses = [], []

        for epoch in range(1, CFG.epochs + 1):
            # ── train ─────────────────────────────────────────────────────────
            model.train()
            t_loss = 0.0
            for xb, yb in train_dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                optimiser.zero_grad()
                pred  = model(xb)
                loss  = criterion(pred, yb)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimiser.step()
                t_loss += loss.item() * len(xb)
            t_loss /= n_train

            # ── validate ──────────────────────────────────────────────────────
            model.eval()
            v_loss = 0.0
            with torch.no_grad():
                for xb, yb in val_dl:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    v_loss += criterion(model(xb), yb).item() * len(xb)
            v_loss /= n_val

            train_losses.append(t_loss)
            val_losses.append(v_loss)
            scheduler.step()
            stopper(v_loss)

            if epoch % 10 == 0 or epoch == 1 or stopper.stop:
                log.info(f"  Epoch {epoch:3d}/{CFG.epochs}  "
                         f"train={t_loss:.5f}  val={v_loss:.5f}  "
                         f"lr={scheduler.get_last_lr()[0]:.2e}")
            if stopper.stop:
                log.info(f"  Early stop at epoch {epoch}.")
                break

        return model, scaler, train_losses, val_losses


# ══════════════════════════════════════════════════════════════════════════════
#  6.  EVALUATION
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_lstm(
        model: "CausalLSTM",
        scaler: StandardScaler,
        test_experiments: List[pd.DataFrame],
) -> Dict:
    """
    Full evaluation on held-out test experiments:
      • Standard regression metrics (RMSE, MAE, R²)
      • Granger ablation: ΔRMSE when voltage channel is zeroed
    """
    log.info("─── Evaluation ──────────────────────────────────────────────")
    model.eval()

    all_true, all_pred, all_pred_noV = [], [], []
    power_mean  = scaler.mean_[1]
    power_scale = scaler.scale_[1]

    for df in test_experiments:
        df_s = df.copy()
        df_s[["input_voltage", "el_power"]] = scaler.transform(
            df[["input_voltage", "el_power"]].values)

        ds = MGTSequenceDataset([df_s], CFG.seq_len)
        dl = TorchLoader(ds, batch_size=512, shuffle=False)

        preds, preds_noV, trues = [], [], []
        with torch.no_grad():
            for xb, yb in dl:
                xb = xb.to(DEVICE)
                preds.append(model(xb).cpu().numpy())
                preds_noV.append(model.granger_ablation(xb, channel=0).cpu().numpy())
                trues.append(yb.numpy())

        p   = np.concatenate(preds)   * power_scale + power_mean
        pnV = np.concatenate(preds_noV)* power_scale + power_mean
        t   = np.concatenate(trues)   * power_scale + power_mean

        all_true.append(t); all_pred.append(p); all_pred_noV.append(pnV)

    true  = np.concatenate(all_true)
    pred  = np.concatenate(all_pred)
    predV = np.concatenate(all_pred_noV)

    rmse  = np.sqrt(mean_squared_error(true, pred))
    mae   = mean_absolute_error(true, pred)
    r2    = r2_score(true, pred)
    rmse_noV = np.sqrt(mean_squared_error(true, predV))
    granger_delta = rmse_noV - rmse     # positive → voltage channel is causal

    log.info(f"  RMSE  : {rmse:.3f} W")
    log.info(f"  MAE   : {mae:.3f} W")
    log.info(f"  R²    : {r2:.4f}")
    log.info(f"  RMSE (voltage zeroed): {rmse_noV:.3f} W  "
             f"(Δ={granger_delta:+.3f} → "
             f"{'voltage IS causal' if granger_delta > 0 else 'voltage NOT causal'})")

    return {
        "rmse": rmse, "mae": mae, "r2": r2,
        "rmse_no_voltage": rmse_noV,
        "granger_delta": granger_delta,
        "true": true, "pred": pred,
    }


# ══════════════════════════════════════════════════════════════════════════════
#  7.  REPORTING & PLOTTING
# ══════════════════════════════════════════════════════════════════════════════

def plot_training_curve(train_losses: List[float],
                        val_losses: List[float], save_path: str) -> None:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.set_facecolor("#0f1117"); fig.patch.set_facecolor("#0f1117")
    ep = range(1, len(train_losses) + 1)
    ax.plot(ep, train_losses, color="#457b9d", lw=2,  label="Train Loss (Huber)")
    ax.plot(ep, val_losses,   color="#e63946", lw=2,  label="Val Loss (Huber)")
    ax.set_xlabel("Epoch", color="white"); ax.set_ylabel("Loss", color="white")
    ax.tick_params(colors="white")
    for sp in ax.spines.values(): sp.set_edgecolor("#444")
    ax.legend(facecolor="#1a1a2e", edgecolor="#555", labelcolor="white")
    ax.set_title("Causal LSTM – Training Curve", color="white", fontsize=11)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()


def plot_predictions(results: Dict, save_path: str, n_points: int = 800) -> None:
    true  = results["true"][:n_points]
    pred  = results["pred"][:n_points]

    fig, axes = plt.subplots(2, 1, figsize=(12, 6))
    for ax in axes:
        ax.set_facecolor("#0f1117"); fig.patch.set_facecolor("#0f1117")

    axes[0].plot(true, color="#2a9d8f", lw=1.2, label="Ground Truth", alpha=0.9)
    axes[0].plot(pred, color="#e63946", lw=1.2, label="Prediction",   alpha=0.8,
                 linestyle="--")
    axes[0].set_title("Electrical Power: Ground Truth vs Prediction",
                      color="white", fontsize=11)
    axes[0].legend(facecolor="#1a1a2e", edgecolor="#555", labelcolor="white")
    axes[0].tick_params(colors="white")
    for sp in axes[0].spines.values(): sp.set_edgecolor("#444")

    residuals = true - pred
    axes[1].plot(residuals, color="#f4a261", lw=1.0, alpha=0.8)
    axes[1].axhline(0, color="#888", lw=0.8, linestyle=":")
    axes[1].set_title("Residuals", color="white", fontsize=11)
    axes[1].tick_params(colors="white")
    for sp in axes[1].spines.values(): sp.set_edgecolor("#444")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close()


def print_summary(results: Dict, causal_results: Optional[Dict] = None) -> None:
    print("\n" + "═"*60)
    print("  MGT CAUSAL PIPELINE — RESULTS SUMMARY")
    print("═"*60)
    if causal_results:
        print(f"  DoWhy ATE  (V → P)  : {causal_results['ate']:+.4f} W / V")
        passed = sum(r["passed"] for r in causal_results["refutation_results"])
        total  = len(causal_results["refutation_results"])
        print(f"  Placebo refutation  : {passed}/{total} trials passed "
              f"({'✓ robust' if passed == total else '⚠ potential overfit'})")
    print(f"  LSTM RMSE           : {results['rmse']:.3f} W")
    print(f"  LSTM MAE            : {results['mae']:.3f} W")
    print(f"  LSTM R²             : {results['r2']:.4f}")
    print(f"  Granger ΔRMSE       : {results['granger_delta']:+.3f} W  "
          f"(voltage causal: {results['granger_delta'] > 0})")
    print("═"*60 + "\n")


# ══════════════════════════════════════════════════════════════════════════════
#  8.  MAIN PIPELINE ORCHESTRATION
# ══════════════════════════════════════════════════════════════════════════════

def run_pipeline():
    log.info("══ MGT Causal Pipeline  START ══════════════════════════════")

    # ── 1. Load data ──────────────────────────────────────────────────────────
    log.info("Stage 1: Loading experiments …")
    train_exps = load_all_experiments("train")
    test_exps  = load_all_experiments("test")
    log.info(f"  {len(train_exps)} train  │  {len(test_exps)} test experiments")

    # ── 2. Causal DAG ─────────────────────────────────────────────────────────
    log.info("Stage 2: Building causal DAG …")
    dag, gml_string = build_causal_dag()
    plot_dag(dag, os.path.join(CFG.output_dir, "causal_dag.png"))

    # ── 3. Regression frame for DoWhy ─────────────────────────────────────────
    log.info("Stage 3: Building regression frame …")
    reg_frame = build_regression_frame(train_exps)

    # ── 4. DoWhy estimation ───────────────────────────────────────────────────
    log.info("Stage 4: DoWhy causal estimation …")
    causal_results = run_dowhy_estimation(reg_frame, gml_string)

    # ── 5. Synthetic interventions (data augmentation) ────────────────────────
    log.info("Stage 5: Generating synthetic interventions …")
    response_surface = ResponseSurface().fit(reg_frame)
    synthetic_exps   = generate_synthetic_interventions(train_exps, response_surface)
    augmented_exps   = train_exps + synthetic_exps
    log.info(f"  Augmented dataset: {len(augmented_exps)} experiments "
             f"({len(train_exps)} real + {len(synthetic_exps)} synthetic)")

    # ── 6. Train Causal LSTM ──────────────────────────────────────────────────
    if not HAS_TORCH:
        log.warning("PyTorch unavailable – skipping LSTM training.")
        return

    log.info("Stage 6: Training Causal LSTM …")
    model, scaler, train_losses, val_losses = train_causal_lstm(augmented_exps)

    # Save model
    ckpt_path = os.path.join(CFG.model_dir, "causal_lstm.pt")
    torch.save({
        "model_state_dict": model.state_dict(),
        "scaler_mean":      scaler.mean_,
        "scaler_scale":     scaler.scale_,
        "config":           CFG.__dict__,
    }, ckpt_path)
    log.info(f"  Model checkpoint → {ckpt_path}")

    # ── 7. Evaluate ───────────────────────────────────────────────────────────
    log.info("Stage 7: Evaluating on test experiments …")
    eval_results = evaluate_lstm(model, scaler, test_exps)

    # ── 8. Plots & summary ────────────────────────────────────────────────────
    log.info("Stage 8: Generating plots …")
    plot_training_curve(train_losses, val_losses,
                        os.path.join(CFG.output_dir, "training_curve.png"))
    plot_predictions(eval_results,
                     os.path.join(CFG.output_dir, "predictions.png"))

    print_summary(eval_results, causal_results)
    log.info("══ MGT Causal Pipeline  DONE ═══════════════════════════════")


# ══════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    run_pipeline()

18:46:53 │ INFO │ ══ MGT Causal Pipeline  START ══════════════════════════════
18:46:53 │ INFO │ Stage 1: Loading experiments …
18:46:53 │ INFO │   Loaded ex_1.csv                       → 9,920 rows  V=[10.  3.]
18:46:53 │ INFO │   Loaded ex_20.csv                      → 6,495 rows  V=[ 3.   5.   7.5 10. ]
18:46:53 │ INFO │   Loaded ex_21.csv                      → 6,495 rows  V=[ 3.   5.   7.5 10. ]
18:46:53 │ INFO │   Loaded ex_23.csv                      → 9,188 rows  V=[3.         3.36842105 3.73684211 4.10526316 4.47368421 4.84210526
 5.21052632 5.57894737 5.94736842 6.31578947 6.68421053 7.05263158
 7.42105263 7.78947368 8.15789474 8.52631579 8.89473684 9.26315789
 9.63157895]
18:46:53 │ INFO │   Loaded ex_24.csv                      → 9,023 rows  V=[ 3.          3.36842105  3.73684211  4.10526316  4.47368421  4.84210526
  5.21052632  5.57894737  5.94736842  6.31578947  6.68421053  7.05263158
  7.42105263  7.78947368  8.15789474  8.52631579  8.89473684  9.26315789
  9.63157895 10

KeyboardInterrupt: 